# Clan-Subnet Detector — Revisión preliminar de rendimiento (Kaggle)

Notebook de **auditoría y rendimiento** del MVP `clan-subnet-detector` (v0.1.0-mvp).

Objetivos:

1. **Correctitud**: toda solución reportada se re-verifica de forma independiente (`is_clique` / `is_independent`).
2. **Rendimiento**: tiempos de construcción de grafo, planificador GWMIN, clique greedy y pipeline de auditoría completo.
3. **Escalado sintético**: grafos tipo Erdős–Rényi y Barabási–Albert en varios tamaños.
4. **Reporte reproducible**: tablas CSV + resumen Markdown + gráfica de tiempos.

**Uso en Kaggle**: Settings → Internet **no es obligatorio** (el código es autocontenido). Ejecute las celdas en orden. `MODE='quick'` reduce tamaños; `MODE='full'` usa el protocolo más amplio.

Resultados se guardan en `/kaggle/working/clan_perf_results/` (o `./clan_perf_results` fuera de Kaggle).


In [ ]:
# CONFIG
MODE = 'quick'          # 'quick' | 'full'
SEED = 42

import os, sys, time, json, platform, math, random
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd

ROOT = Path('/kaggle/working') if Path('/kaggle').exists() else Path.cwd()
OUT = ROOT / 'clan_perf_results'
OUT.mkdir(parents=True, exist_ok=True)
SRC = ROOT / 'clan_src'
SRC.mkdir(exist_ok=True)

if MODE == 'quick':
    N_LIST_ER = [200, 500, 1000, 2000]
    N_LIST_BA = [500, 1000, 2000, 4000]
    REPS = 3
    AVG_DEG = 6
elif MODE == 'full':
    N_LIST_ER = [200, 500, 1000, 2000, 5000, 10000]
    N_LIST_BA = [500, 1000, 2000, 5000, 10000, 20000]
    REPS = 5
    AVG_DEG = 6
else:
    raise ValueError("MODE debe ser 'quick' o 'full'")

print('Python', sys.version.split()[0], '|', platform.platform(), '| CPUs:', os.cpu_count())
print('MODE =', MODE, '| OUT =', OUT)


## 1. Código fuente del MVP (autocontenido)

Se escribe el núcleo del paquete en `clan_src/` para que el notebook no dependa del repositorio privado.


In [ ]:
# --- graph.py ---
(SRC / 'graph.py').write_text(r"""
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Dict, Iterable, List, Optional, Set, Tuple

@dataclass(frozen=True)
class Edge:
    u: int
    v: int
    weight: float = 1.0

@dataclass
class Graph:
    n: int
    directed: bool = False
    adj: List[Dict[int, float]] = field(default_factory=list)
    node_attrs: Dict[int, dict] = field(default_factory=dict)

    def __post_init__(self) -> None:
        if not self.adj:
            self.adj = [{} for _ in range(self.n)]
        if len(self.adj) != self.n:
            raise ValueError("adj length must equal n")

    @classmethod
    def from_edges(cls, n: int, edges: Iterable, directed: bool = False) -> "Graph":
        g = cls(n=n, directed=directed)
        for e in edges:
            if isinstance(e, Edge):
                g.add_edge(e.u, e.v, e.weight)
            elif len(e) == 2:
                g.add_edge(e[0], e[1])
            else:
                g.add_edge(e[0], e[1], e[2])
        return g

    def add_edge(self, u: int, v: int, weight: float = 1.0) -> None:
        if not (0 <= u < self.n and 0 <= v < self.n):
            raise IndexError("node index out of range")
        self.adj[u][v] = self.adj[u].get(v, 0.0) + weight
        if not self.directed and u != v:
            self.adj[v][u] = self.adj[v].get(u, 0.0) + weight

    def neighbors(self, u: int) -> Dict[int, float]:
        return self.adj[u]

    def degree(self, u: int) -> int:
        return len(self.adj[u])

    def edge_count(self) -> int:
        total = sum(len(neigh) for neigh in self.adj)
        return total if self.directed else total // 2

    def induced_subgraph(self, nodes: Iterable[int]) -> "Graph":
        node_list = sorted(set(nodes))
        mapping = {old: new for new, old in enumerate(node_list)}
        sub = Graph(n=len(node_list), directed=self.directed)
        for old_u in node_list:
            for old_v, w in self.adj[old_u].items():
                if old_v in mapping:
                    sub.add_edge(mapping[old_u], mapping[old_v], w)
        return sub

    def has_edge(self, u: int, v: int) -> bool:
        return v in self.adj[u]
""")

# --- metrics.py ---
(SRC / 'metrics.py').write_text(r"""
from __future__ import annotations
from collections import deque
from typing import List, Set
from graph import Graph

def density(g: Graph) -> float:
    m = g.edge_count()
    n = g.n
    if n < 2:
        return 0.0
    max_edges = n * (n - 1) if g.directed else n * (n - 1) // 2
    return m / max_edges if max_edges else 0.0

def connected_components(g: Graph) -> List[Set[int]]:
    visited = [False] * g.n
    components: List[Set[int]] = []
    for start in range(g.n):
        if visited[start]:
            continue
        comp: Set[int] = set()
        queue = deque([start])
        visited[start] = True
        while queue:
            u = queue.popleft()
            comp.add(u)
            for v in g.adj[u]:
                if not visited[v]:
                    visited[v] = True
                    queue.append(v)
            if g.directed:
                for v in range(g.n):
                    if u in g.adj[v] and not visited[v]:
                        visited[v] = True
                        queue.append(v)
        components.append(comp)
    return components
""")

# --- independent_set.py ---
(SRC / 'independent_set.py').write_text(r"""
from __future__ import annotations
from typing import Iterable, List, Optional, Sequence, Set
from graph import Graph

def gwmin_independent(
    g: Graph,
    banned: Optional[Iterable[int]] = None,
    k: Optional[int] = None,
    weights: Optional[Sequence[float]] = None,
) -> List[int]:
    n = g.n
    banned_set: Set[int] = set(banned or [])
    alive = [i not in banned_set for i in range(n)]
    w = list(weights) if weights is not None else [1.0] * n
    if len(w) != n:
        raise ValueError("weights length must equal number of nodes")
    deg = [0] * n
    for u in range(n):
        if not alive[u]:
            continue
        for v in g.adj[u]:
            if alive[v]:
                deg[u] += 1
    picked: List[int] = []
    while True:
        if k is not None and len(picked) >= k:
            break
        best = -1
        best_score = -1.0
        for v in range(n):
            if not alive[v]:
                continue
            score = w[v] / (deg[v] + 1)
            if score > best_score:
                best_score = score
                best = v
        if best < 0:
            break
        picked.append(best)
        alive[best] = False
        for u in list(g.adj[best].keys()):
            if alive[u]:
                alive[u] = False
                for x in g.adj[u]:
                    if alive[x]:
                        deg[x] -= 1
        if g.directed:
            for u in range(n):
                if alive[u] and best in g.adj[u]:
                    alive[u] = False
                    for x in g.adj[u]:
                        if alive[x]:
                            deg[x] -= 1
    return picked

def is_independent(g: Graph, nodes: Iterable[int]) -> bool:
    node_set = set(nodes)
    for u in node_set:
        for v in g.adj[u]:
            if v in node_set and u != v:
                return False
    return True
""")

# --- clique.py ---
(SRC / 'clique.py').write_text(r"""
from __future__ import annotations
from typing import Iterable, List, Set
from graph import Graph

def is_clique(g: Graph, nodes: Iterable[int]) -> bool:
    node_list = list(nodes)
    for i, u in enumerate(node_list):
        for v in node_list[i + 1:]:
            if g.directed:
                if not (g.has_edge(u, v) or g.has_edge(v, u)):
                    return False
            else:
                if not g.has_edge(u, v):
                    return False
    return True

def greedy_clique(g: Graph, seed: int | None = None) -> List[int]:
    if g.n == 0:
        return []
    if seed is None:
        seed = max(range(g.n), key=lambda u: g.degree(u))
    clique: List[int] = [seed]
    candidates: Set[int] = set(g.adj[seed].keys())
    while candidates:
        best = max(candidates, key=lambda u: sum(1 for v in candidates if g.has_edge(u, v)))
        if all(g.has_edge(best, c) or (g.directed and g.has_edge(c, best)) for c in clique):
            clique.append(best)
            candidates &= set(g.adj[best].keys())
        else:
            candidates.discard(best)
    return clique
""")

# --- scoring.py ---
(SRC / 'scoring.py').write_text(r"""
from __future__ import annotations
from typing import Iterable
from graph import Graph
from metrics import density

def score_clique(g: Graph, nodes: Iterable[int], base_weight: float = 1.0) -> float:
    node_list = list(nodes)
    k = len(node_list)
    if k < 2:
        return 0.0
    sub = g.induced_subgraph(node_list)
    dens = density(sub)
    avg_deg = sum(g.degree(u) for u in node_list) / k
    return base_weight * (k ** 1.5) * dens * (1.0 + 0.1 * avg_deg)

def score_independent_set(g: Graph, nodes: Iterable[int], base_weight: float = 1.0) -> float:
    node_list = list(nodes)
    k = len(node_list)
    if k == 0:
        return 0.0
    node_set = set(node_list)
    cut = 0
    for u in node_list:
        for v in g.adj[u]:
            if v not in node_set:
                cut += 1
    isolation = 1.0 / (1.0 + cut / max(k, 1))
    return base_weight * k * isolation
""")

print('Fuente MVP escrita en', SRC)


In [ ]:
# Importar el núcleo escrito arriba
sys.path.insert(0, str(SRC))
from graph import Graph
from metrics import density, connected_components
from independent_set import gwmin_independent, is_independent
from clique import greedy_clique, is_clique
from scoring import score_clique, score_independent_set

def erdos_renyi(n: int, avg_deg: float, seed: int) -> Graph:
    rng = random.Random(seed)
    p = avg_deg / max(n - 1, 1)
    edges = []
    for i in range(n):
        for j in range(i + 1, n):
            if rng.random() < p:
                edges.append((i, j))
    return Graph.from_edges(n, edges, directed=False)

def barabasi_albert(n: int, m: int, seed: int) -> Graph:
    """BA preferential attachment (m edges per new node)."""
    rng = random.Random(seed)
    if n <= m:
        edges = [(i, j) for i in range(n) for j in range(i+1, n)]
        return Graph.from_edges(n, edges, directed=False)
    edges = [(i, j) for i in range(m + 1) for j in range(i + 1, m + 1)]
    deg = [m] * (m + 1) + [0] * (n - m - 1)
    for new in range(m + 1, n):
        targets = set()
        total = sum(deg[:new])
        while len(targets) < m and total > 0:
            r = rng.random() * total
            acc = 0.0
            for v in range(new):
                acc += deg[v]
                if acc >= r:
                    targets.add(v)
                    break
        for t in targets:
            edges.append((new, t))
            deg[new] += 1
            deg[t] += 1
    return Graph.from_edges(n, edges, directed=False)

print('Import OK | generadores listos')


## 2. Verificación de correctitud (smoke)

Antes de medir tiempos se confirma que los verifiers y el pipeline básico funcionan.


In [ ]:
def smoke_check():
    edges = [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3),(0,4),(3,5),(7,8),(8,9)]
    g = Graph.from_edges(10, edges, directed=False)
    cl = greedy_clique(g)
    ind = gwmin_independent(g, k=6)
    assert is_clique(g, cl), 'clique verification failed'
    assert is_independent(g, ind), 'independent-set verification failed'
    assert density(g) > 0
    comps = connected_components(g)
    assert len(comps) >= 1
    print(f'SMOKE PASS | clique={cl} (size={len(cl)}) | ind={ind} (size={len(ind)}) | components={len(comps)}')

smoke_check()


## 3. Benchmark de rendimiento

Se miden, por cada tamaño de grafo:

- tiempo de construcción del grafo
- tiempo de `gwmin_independent` (k ilimitado y k=10)
- tiempo de `greedy_clique`
- tiempo del pipeline de auditoría simplificado (detect + verify + score)
- flags de correctitud


In [ ]:
def audit_pipeline(g: Graph, k: int = 10):
    t0 = time.perf_counter()
    cl = greedy_clique(g)
    cl_ok = is_clique(g, cl)
    sc_cl = score_clique(g, cl)
    ind = gwmin_independent(g, k=k)
    ind_ok = is_independent(g, ind)
    sc_ind = score_independent_set(g, ind)
    comps = connected_components(g)
    dens = density(g)
    elapsed = (time.perf_counter() - t0) * 1000
    return {
        'clique_size': len(cl),
        'clique_ok': cl_ok,
        'clique_score': sc_cl,
        'ind_size': len(ind),
        'ind_ok': ind_ok,
        'ind_score': sc_ind,
        'n_components': len(comps),
        'density': dens,
        'audit_ms': elapsed,
    }

def median(xs):
    s = sorted(xs)
    return s[len(s)//2]

rows = []
for family, n_list, builder in [
    ('ER', N_LIST_ER, lambda n, seed: erdos_renyi(n, AVG_DEG, seed)),
    ('BA', N_LIST_BA, lambda n, seed: barabasi_albert(n, max(2, AVG_DEG//2), seed)),
]:
    for n in n_list:
        build_t, gwmin_t, gwmin_k_t, clique_t, audit_t = [], [], [], [], []
        clique_sizes, ind_sizes = [], []
        all_ok = True
        m_edges = 0
        for r in range(REPS):
            seed = SEED + r * 1009 + n
            t0 = time.perf_counter()
            g = builder(n, seed)
            build_t.append((time.perf_counter() - t0) * 1000)
            m_edges = g.edge_count()

            t0 = time.perf_counter()
            ind_full = gwmin_independent(g)
            gwmin_t.append((time.perf_counter() - t0) * 1000)
            if not is_independent(g, ind_full):
                all_ok = False

            t0 = time.perf_counter()
            ind_k = gwmin_independent(g, k=10)
            gwmin_k_t.append((time.perf_counter() - t0) * 1000)

            t0 = time.perf_counter()
            cl = greedy_clique(g)
            clique_t.append((time.perf_counter() - t0) * 1000)
            if not is_clique(g, cl):
                all_ok = False

            res = audit_pipeline(g, k=10)
            audit_t.append(res['audit_ms'])
            if not (res['clique_ok'] and res['ind_ok']):
                all_ok = False
            clique_sizes.append(res['clique_size'])
            ind_sizes.append(res['ind_size'])

        rows.append({
            'family': family,
            'n': n,
            'm': m_edges,
            'avg_deg_approx': round(2 * m_edges / max(n, 1), 2),
            'build_ms_med': median(build_t),
            'gwmin_full_ms_med': median(gwmin_t),
            'gwmin_k10_ms_med': median(gwmin_k_t),
            'clique_ms_med': median(clique_t),
            'audit_ms_med': median(audit_t),
            'clique_size_med': median(clique_sizes),
            'ind_size_med': median(ind_sizes),
            'all_verified': all_ok,
            'reps': REPS,
        })
        print(f"[{family}] n={n:<6d} m≈{m_edges:<7d} build={median(build_t):7.1f}ms  "
              f"gwmin={median(gwmin_t):7.1f}ms  clique={median(clique_t):6.1f}ms  "
              f"audit={median(audit_t):7.1f}ms  verified={all_ok}")

perf = pd.DataFrame(rows)
perf.to_csv(OUT / 'performance_scaling.csv', index=False)
display(perf.round(2))


## 4. Gráficas de escalado


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, family in zip(axes, ['ER', 'BA']):
    sub = perf[perf.family == family]
    if sub.empty:
        continue
    ax.plot(sub.n, sub.gwmin_full_ms_med, 'o-', label='GWMIN full')
    ax.plot(sub.n, sub.gwmin_k10_ms_med, 's--', label='GWMIN k=10')
    ax.plot(sub.n, sub.clique_ms_med, '^-', label='Greedy clique')
    ax.plot(sub.n, sub.audit_ms_med, 'd-', label='Audit pipeline')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('n (nodos)')
    ax.set_ylabel('Tiempo mediana (ms)')
    ax.set_title(f'{family} — tiempos vs n')
    ax.grid(True, which='both', alpha=0.3)
    ax.legend(fontsize=8)

fig.tight_layout()
fig.savefig(OUT / 'scaling_times.png', dpi=150)
plt.show()
print('Figura guardada:', OUT / 'scaling_times.png')


## 5. Resumen de correctitud y veredicto preliminar


In [ ]:
checks = {
    'all_rows_verified': bool(perf.all_verified.all()),
    'no_empty_graphs': bool((perf.n > 0).all()),
    'clique_sizes_positive': bool((perf.clique_size_med >= 1).all()),
    'independent_sizes_nonneg': bool((perf.ind_size_med >= 0).all()),
}

worst = perf.sort_values('audit_ms_med', ascending=False).head(5)[
    ['family', 'n', 'm', 'audit_ms_med', 'gwmin_full_ms_med', 'clique_ms_med', 'all_verified']
]

print('=== Automated checks ===')
for k, v in checks.items():
    print(f'  {k}: {"PASS" if v else "FAIL"}')

print('\n=== Top 5 tiempos de auditoría (ms) ===')
display(worst.round(2))

report = {
    'mode': MODE,
    'seed': SEED,
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'cpu_count': os.cpu_count(),
    'avg_deg_target': AVG_DEG,
    'reps': REPS,
    'checks': checks,
    'n_rows': len(perf),
    'max_n': int(perf.n.max()),
    'max_audit_ms_med': float(perf.audit_ms_med.max()),
    'families': sorted(perf.family.unique().tolist()),
}
(OUT / 'perf_report.json').write_text(json.dumps(report, indent=2))

md_lines = [
    '# Clan-Subnet Detector — Performance Report',
    '',
    f'- Mode: `{MODE}`',
    f'- Seed: {SEED}',
    f'- Python: {report["python"]}',
    f'- Platform: {report["platform"]}',
    f'- CPUs: {report["cpu_count"]}',
    f'- Max n tested: {report["max_n"]}',
    f'- Max audit median (ms): {report["max_audit_ms_med"]:.1f}',
    '',
    '## Automated checks',
]
for k, v in checks.items():
    md_lines.append(f'- `{k}`: **{"PASS" if v else "FAIL"}**')
md_lines += [
    '',
    '## Scaling table',
    perf.round(2).to_markdown(index=False),
    '',
    '## Notes',
    '- Timings are medians over the configured number of repetitions.',
    '- Every reported clique and independent set was re-verified independently.',
    '- GWMIN is a greedy stand-in; a MORPH binary can replace it later without changing the audit contract.',
    '',
]
(OUT / 'PERF_REPORT.md').write_text('\n'.join(md_lines))
print('\nReportes guardados en', OUT)
print('Archivos:', *[p.name for p in sorted(OUT.iterdir())], sep='\n- ')


## 6. Interpretación rápida (orientación)

| Observación | Qué mirar |
|-------------|-----------|
| `all_rows_verified = PASS` | Correctitud del núcleo en todos los tamaños probados |
| Curva `gwmin_full_ms` | Coste del planificador de conjuntos independientes (dominante en grafos densos) |
| `gwmin_k10` vs full | Beneficio de limitar k cuando solo se necesitan respaldos / top-k |
| `audit_ms` | Coste de un ciclo completo detect+verify+score (métrica operativa del MVP) |
| Diferencia ER vs BA | Sensibilidad a la heterogeneidad de grados |

Este notebook no pretende ser un benchmark competitivo exhaustivo; es una **revisión preliminar** de rendimiento y de la garantía de verificación del MVP.

**Siguiente paso sugerido**: cambiar `MODE = 'full'` y re-ejecutar en una sesión con más CPU/RAM, o sustituir el planificador GWMIN por una llamada al binario MORPH para comparar calidad y tiempo.
